In [2]:
import torch
import torch.nn as nn
import torchinfo
from model.mini_cnn import MiniCNN
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

In [3]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

✓ Using Apple Silicon GPU (MPS)


In [4]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()
    return accuracies, losses

In [5]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from utils.CustomDataset import CustomDataset
from utils.CustomSkibidiDataset import CustomSkibidiDataset

from torchvision.transforms import v2

In [ ]:
# 1. Define Image Transforms
# HUUUUGOOOOOOOO
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     torchvision.transforms.functional.to_dtype(float16, scale=True),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


#Initialize datasets
hierarchical_dataset_1 = CustomSkibidiDataset(
    img_dir='data/hierarchical_level1/',
    csv_mapper='data/hierarchical_level1.csv',
    transform=transform, 
    target_transform=None # ???
)

hierarchical_dataset_2 = CustomSkibidiDataset(
    img_dir='data/hierarchical_level2/',
    csv_mapper='data/hierarchical_level2.csv',
    transform=transform, 
    target_transform=None # ???
)

hierarchical_dataset_3 = CustomSkibidiDataset(
    img_dir='data/hierarchical_level3/',
    csv_mapper='data/hierarchical_level3.csv',
    transform=transform, 
    target_transform=None # ???
)

#Train test split 
train_size_1 = int(0.8 * len(hierarchical_dataset_1))
test_size_1 = len(hierarchical_dataset_1) - train_size_1
train_dataset_1, validation_dataset_1 = random_split(hierarchical_dataset_1, [train_size_1, test_size_1])

train_size_2 = int(0.8 * len(hierarchical_dataset_2))
test_size_2 = len(hierarchical_dataset_2) - train_size_2
train_dataset_2, validation_dataset_2 = random_split(hierarchical_dataset_2, [train_size_2, test_size_2])

train_size_3 = int(0.8 * len(hierarchical_dataset_3))
test_size_3 = len(hierarchical_dataset_3) - train_size_3
train_dataset_3, validation_dataset_3 = random_split(hierarchical_dataset_3, [train_size_3, test_size_3])

Found folders ['Andrena hattorfiana', 'Andrena fulvata', 'Andrena haemorrhoa', 'Andrena leucophaea', 'Andrena banksi', 'Andrena dorsata', 'Andrena florivaga', 'Andrena barbareae', 'Andrena convallaria', 'Andrena fulva', 'Andrena nitida', 'Andrena lineolata', 'Andrena bicolor', 'Andrena fortipunctata', 'Andrena villipes', 'Andrena denticulata', 'Andrena angustitarsata', 'Andrena mendica', 'Andrena aerinifrons', 'Andrena costillensis', 'Andrena chengtehensis', 'Andrena wilkella', 'Andrena irana', 'Andrena discors', 'Andrena barbilabris', 'Andrena plana', 'Andrena prodigiosa', 'Andrena orbitalis', 'Andrena nigroaenea', 'Andrena ventralis', 'Andrena rudbeckiae', 'Andrena vaga', 'Andrena flavipes', 'Andrena pinguis', 'Andrena perplexa', 'Andrena aliciae', 'Andrena mediovittata', 'Andrena melanochroa', 'Andrena cineraria', 'Andrena vulpecula', 'Andrena rufula', 'Andrena afrensis', 'Andrena carantonica', 'Andrena hesperia', 'Andrena vicinoides', 'Andrena limbata', 'Andrena dilleri', 'Andrena 

#TO DO - Faire 3 modèles entraînés sur les 3 datasets

In [ ]:
#set up training
epochs = 20
model1 = MiniCNN()
model1.to(device)

train_set1 = DataLoader(
    dataset=train_dataset_1,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

validation_test1 = DataLoader(
    dataset=validation_dataset_1,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

Model training | Epoch 1/5:   0%|          | 0/656 [00:00<?, ?it/s]/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Model training | Epoch 1/5: 100%|██████████| 656/656 [03:24<00:00,  3.21it/s, batch_loss=3.2997, accuracy=9.96%]
Model training | Epoch 2/5: 100%|██████████| 656/656 [03:36<00:00,  3.03it/s, batch_loss=2.8967, accuracy=19.50%]
Model training | Epoch 3/5: 100%|██████████| 656/656 [04:17<00:00,  2.55it/s, batch_loss=3.2141, accuracy=27.56%]  
Model training | Epoch 4/5: 100%|██████████| 656/656 [03:40<00:00,  2.97it/s, batch_loss=1.6622, accuracy=34.27%]
Model training | Epoch 5/5: 100%|██████████| 656/656 [03:32<00:00,  3.08it/s, batch_loss=1.6849, accuracy=40.92%]


In [ ]:
#set up training
epochs = 20
model2 = MiniCNN()
model2.to(device)

train_set2 = DataLoader(
    dataset=train_dataset_2,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

validation_test2 = DataLoader(
    dataset=validation_dataset_2,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

In [ ]:
#set up training
epochs = 20
model3 = MiniCNN()
model3.to(device)

train_set3 = DataLoader(
    dataset=train_dataset_3,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

validation_test3 = DataLoader(
    dataset=validation_dataset_3,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

In [ ]:
#training
training_curves, training_curves = train_crossEntropyLoss(train_set1,model1, epochs)
training_curves, training_curves = train_crossEntropyLoss(train_set2,model2, epochs)
training_curves, training_curves = train_crossEntropyLoss(train_set3,model3, epochs)

In [ ]:
torch.save(model1.state_dict(), f"model/trained_model/Hierarchical1_cnn.pth")
torch.save(model2.state_dict(), f"model/trained_model/Hierarchical2_cnn.pth")
torch.save(model3.state_dict(), f"model/trained_model/Hierarchical3_cnn.pth")

In [14]:
import csv

In [ ]:
# import torch
# from tqdm import tqdm
# import csv

# def hierarchical_prediction_to_csv(validation_dataset, model1, model2, model3, device, threshold1=0.0, threshold2=0.0):
#     """
#     Fait l'inference hiérarchique sur trois modèles :
#     - model1 : classes fortes
#     - model2 : classes moyennes
#     - model3 : classes rares
#     Écrit les résultats dans prediction_hierarchical.csv
#     """

#     models = [model1, model2, model3]
#     for m in models:
#         m.to(device)
#         m.eval()

#     id_counter = 0
#     predictions_array = []

#     with torch.no_grad():
#         pbar = tqdm(validation_dataset, desc="Hierarchical Inference")
#         for images, labels in pbar:

#             images = images.to(device)
#             batch_size = images.shape[0]

#             # on initialise les images à traiter par le modèle 1
#             images_to_process = images
#             final_preds = torch.zeros(batch_size, dtype=torch.int32).to(device)
#             remaining_mask = torch.ones(batch_size, dtype=torch.bool).to(device)

#             # Parcours hiérarchique
#             for level, model in enumerate(models):
#                 if remaining_mask.sum() == 0:
#                     break  # toutes les images ont été prédictes

#                 current_images = images_to_process[remaining_mask]
#                 logits = model(current_images)
#                 preds = logits.argmax(dim=1)

#                 # seuil optionnel pour rerouter vers le prochain niveau
#                 if level == 0:
#                     mask_next = (preds == 99) | (logits.max(dim=1).values < threshold1)
#                 elif level == 1:
#                     mask_next = (preds == 99) | (logits.max(dim=1).values < threshold2)
#                 else:
#                     mask_next = torch.zeros_like(preds, dtype=torch.bool)  # dernier niveau, tout est accepté

#                 # remplir les predictions finales pour celles qui ne vont pas au niveau suivant
#                 idx_final = remaining_mask.nonzero(as_tuple=True)[0][~mask_next]
#                 final_preds[idx_final] = preds[~mask_next]

#                 # mise à jour du masque pour le prochain niveau
#                 remaining_mask = mask_next

#             # enregistrement batch
#             for pred, label in zip(final_preds.cpu().numpy(), labels):
#                 id_counter += 1
#                 predictions_array.append({
#                     "id": id_counter,
#                     "pred": int(pred),
#                     "true": int(label)
#                 })

#     # sauvegarde CSV
#     output_path = "prediction_hierarchical.csv"
#     with open(output_path, "w", newline='', encoding="utf-8") as f:
#         writer = csv.DictWriter(f, fieldnames=["id", "pred", "true"])
#         writer.writeheader()
#         writer.writerows(predictions_array)

#     print(f"Les predictions hiérarchiques ont été sauvegardées dans {output_path}.")

In [ ]:
import pandas as pd


def load_label_mapping(mapping_csv_path):
    """
    Retourne :
    - hier_to_original : dict[int → int]
    - others_hier_label : int ou None
    """
    df = pd.read_csv(mapping_csv_path)

    hier_to_original = dict(
        zip(df["hier_label"], df["original_label"])
    )

    others_rows = df[df["original_label"] == 99]
    others_hier_label = (
        int(others_rows["hier_label"].iloc[0])
        if len(others_rows) > 0 else None
    )

    return hier_to_original, others_hier_label

In [ ]:
import torch
import csv
from tqdm import tqdm


def hierarchical_prediction_to_csv(
    validation_dataset,
    model1,
    model2,
    model3,
    device,
    mapping_lvl1_csv,
    mapping_lvl2_csv,
    mapping_lvl3_csv,
    threshold1=0.0,
    threshold2=0.0,
    output_path="prediction_hierarchical.csv"
):
    """
    Inference hiérarchique à 3 niveaux avec remapping via CSV
    """

    # ===============================
    # Load mappings
    # ===============================
    hier_to_orig_1, others_lvl1 = load_label_mapping(mapping_lvl1_csv)
    hier_to_orig_2, others_lvl2 = load_label_mapping(mapping_lvl2_csv)
    hier_to_orig_3, _ = load_label_mapping(mapping_lvl3_csv)  # no others lvl3

    models = [model1, model2, model3]
    for m in models:
        m.to(device)
        m.eval()

    predictions_array = []
    id_counter = 0

    with torch.no_grad():
        pbar = tqdm(validation_dataset, desc="Hierarchical inference")

        for images, labels in pbar:
            images = images.to(device)
            batch_size = images.size(0)

            remaining = torch.ones(batch_size, dtype=torch.bool, device=device)
            final_preds = torch.full(
                (batch_size,), -1, dtype=torch.long, device=device
            )

            # ===============================
            # LEVEL 1
            # ===============================
            if remaining.any():
                imgs = images[remaining]
                logits = model1(imgs)
                conf, preds = logits.max(dim=1)

                to_next = (preds == others_lvl1) | (conf < threshold1)
                idx_global = remaining.nonzero(as_tuple=True)[0]

                for i, p in zip(idx_global[~to_next], preds[~to_next]):
                    final_preds[i] = hier_to_orig_1[int(p)]

                remaining[idx_global] = to_next

            # ===============================
            # LEVEL 2
            # ===============================
            if remaining.any():
                imgs = images[remaining]
                logits = model2(imgs)
                conf, preds = logits.max(dim=1)

                to_next = (preds == others_lvl2) | (conf < threshold2)
                idx_global = remaining.nonzero(as_tuple=True)[0]

                for i, p in zip(idx_global[~to_next], preds[~to_next]):
                    final_preds[i] = hier_to_orig_2[int(p)]

                remaining[idx_global] = to_next

            # ===============================
            # LEVEL 3 (final)
            # ===============================
            if remaining.any():
                imgs = images[remaining]
                preds = model3(imgs).argmax(dim=1)
                idx_global = remaining.nonzero(as_tuple=True)[0]

                for i, p in zip(idx_global, preds):
                    final_preds[i] = hier_to_orig_3[int(p)]

            # ===============================
            # CSV logging
            # ===============================
            for pred, true in zip(final_preds.cpu().numpy(), labels):
                id_counter += 1
                predictions_array.append({
                    "id": id_counter,
                    "pred": int(pred),
                    "true": int(true)
                })

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "pred", "true"])
        writer.writeheader()
        writer.writerows(predictions_array)

    print(f"CSV hiérarchique sauvegardé : {output_path}")

In [ ]:
hierarchical_prediction_to_csv(validation_dataset_1, model1, model2, model3, device)

Inference:   0%|          | 0/164 [00:00<?, ?it/s]/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Inference: 100%|██████████| 164/164 [00:39<00:00,  4.17it/s]

Les predictions ont été sauvegardees dans prediction.csv.


In [28]:
import pandas as pd
from eval.Analyser import Analyser

In [ ]:
data = pd.read_csv("prediction_hierarchical.csv")

analyser = Analyser(data)
report = analyser.generate_report()
print(report)

{'f1_score_avg': 0.28708540208615774, 'f1_score_per_class': array([0.30769231, 0.0483871 , 0.72340426, 0.32542373, 0.02352941,
       0.05405405, 0.64761905, 0.61654135, 0.28865979, 0.14285714,
       0.25930101, 0.        , 0.        , 0.22222222, 0.32490975,
       0.53623188, 0.16748768, 0.2893617 , 0.06802721, 0.51282051,
       0.4375    , 0.07246377, 0.52475248, 0.17748344, 0.29559748,
       0.19444444, 0.        , 0.61702128, 0.27906977, 0.58823529,
       0.05333333, 0.15873016, 0.0754717 , 0.28169014, 0.10891089,
       0.6342711 , 0.29565217, 0.29239766, 0.26515152, 0.08484848,
       0.28888889, 0.30120482, 0.53676471, 0.04968944, 0.22754491,
       0.31683168, 0.2195122 ]), 'best_f1': np.int64(2)}
